# Occupation Signals Before Polity Collapse: A Difference-in-Differences Approach

The relative prevalence of specific professional groups within a society can serve as an indicator of institutional development and, in some cases, as a precursor to periods of social or political disruption (Turchin, 2018). The Cultura database enables the computation of the proportion of individuals recorded under a given occupation within a polity or region over time.

**Methodology:**

1. We divide terminated polities into **short-lived** (< 150 years) and **long-lasting** (≥ 150 years). For short-lived polities, we compare occupation shares **25 years before** and **at the time of collapse**; for long-lasting polities, **100 years before** and **at the time of collapse**.
2. We require at least **100 recorded individuals** per polity and at least **10 individuals with a specific occupation** to avoid noise. We use the polity's end date as the collapse date and exclude all modern polities.
3. We apply a **Wilcoxon signed-rank test** to detect systematic changes in occupation share across polities.
4. To isolate collapse-specific signals from broader secular trends, we apply a **difference-in-differences** design, comparing each collapsed polity against its **5 geographically closest peers**.
5. We plot the polities with the largest occupation-share changes alongside their top 3 regional neighbours.

## 1. Data Loading

We load all terminated (non-modern) polities from `cliopatria_polity_periods`, classify them by duration, and collect occupation shares in two comparison windows per polity type:
- **Short-lived (< 150 yr):** crisis window [t−25, t] vs. pre-crisis window [t−50, t−25]
- **Long-lasting (≥ 150 yr):** crisis window [t−100, t] vs. pre-crisis window [t−200, t−100]

We also compute polity centroids (mean lat/lon of matched individuals) to identify the 5 nearest geographic neighbours for the difference-in-differences control.

In [50]:
import sqlite3, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter, defaultdict
from scipy.stats import wilcoxon
from scipy.spatial.distance import cdist

DB = '../data/humans_clean.sqlite3'
conn = sqlite3.connect(DB)
conn.execute('PRAGMA cache_size=-500000')

In [51]:
# !pip install pyarrow fastparquet

In [52]:
# ── Parameters ──
DURATION_THRESHOLD = 150   # years: long-lived if >= 150
MIN_DURATION       = 26    # minimum polity duration (exclude < 26 years)
MIN_INDIVIDUALS    = 100   # minimum individuals per polity
MODERN_CUTOFF      = 2000  # exclude polities ending after this year

# Time windows (decade-based)
CRISIS_WINDOW      = 10    # crisis = last decade before collapse
PRE_CRISIS_GAP     = 20    # pre-crisis starts 20 years before collapse (2 decades before crisis)
PRE_CRISIS_WINDOW  = 10    # pre-crisis = 1 decade window

import os
from tqdm import tqdm

CHECKPOINT_DIR = '../checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def save_checkpoint(name, df):
    df.to_pickle(f'{CHECKPOINT_DIR}/{name}.pkl')
    print(f"  ✓ Saved: {name}.pkl")

def load_checkpoint(name):
    path = f'{CHECKPOINT_DIR}/{name}.pkl'
    if os.path.exists(path):
        print(f"  ✓ Loaded: {name}.pkl")
        return pd.read_pickle(path)
    return None

def extract_lowest_level_polity(polity_chain):
    """Extract the lowest-level polity from a hierarchy chain."""
    segments = polity_chain.split(';')
    for seg in reversed(segments):
        seg = seg.strip()
        if not seg.startswith('(') and not seg.endswith(')'):
            return seg
    return None

def get_lowest_level_polity_id(polity_chain, polity_id_chain):
    """Get the polity_id corresponding to the lowest-level polity."""
    segments = polity_chain.split(';')
    ids = polity_id_chain.split(';')
    for i in range(len(segments) - 1, -1, -1):
        seg = segments[i].strip()
        if not seg.startswith('(') and not seg.endswith(')'):
            return int(ids[i]) if i < len(ids) else int(ids[-1])
    return int(ids[-1])

def get_largest_polygon_centroid(geom):
    """Extract centroid from the largest polygon in a geometry."""
    gtype = geom.get('type', '')
    if gtype == 'Polygon':
        coords = geom['coordinates'][0]
        return np.mean([c[1] for c in coords]), np.mean([c[0] for c in coords])
    elif gtype == 'MultiPolygon':
        largest_poly = max(geom['coordinates'], key=lambda p: len(p[0]))[0]
        return np.mean([c[1] for c in largest_poly]), np.mean([c[0] for c in largest_poly])
    return None, None

# ── 1a. Load raw data ──
df_raw = load_checkpoint('df_raw')
if df_raw is None:
    print("Loading data from database...")
    df_raw = pd.read_sql_query('''
        SELECT wikidata_id, polity_name, polity_id, impact_year
        FROM consolidate
        WHERE impact_year IS NOT NULL AND occupations IS NOT NULL
    ''', conn)
    print(f"Loaded {len(df_raw):,} individuals")

    print("Extracting lowest-level polities...")
    df_raw['lowest_polity'] = df_raw['polity_name'].apply(extract_lowest_level_polity)
    df_raw['lowest_pid'] = df_raw.apply(lambda r: get_lowest_level_polity_id(r['polity_name'], r['polity_id']), axis=1)
    df_raw = df_raw[df_raw['lowest_polity'].notna()].copy()
    
    save_checkpoint('df_raw', df_raw)

print(f'Individuals: {len(df_raw):,}')

# ── 1b. Get polity metadata ──
df_polity_meta = load_checkpoint('df_polity_meta')
if df_polity_meta is None:
    print("Building polity metadata...")
    
    # Aggregate by lowest_polity
    lowest_level_counts = df_raw.groupby('lowest_polity').agg(
        count=('wikidata_id', 'count'),
        max_pid=('lowest_pid', 'max')
    ).reset_index()
    
    # Get time periods from database
    all_pids = lowest_level_counts['max_pid'].unique().tolist()
    pid_periods = {}
    for row in conn.execute(f'''
        SELECT polity_id, MIN(from_year), MAX(to_year) 
        FROM cliopatria_polity_periods 
        WHERE polity_id IN ({','.join(map(str, all_pids))})
        GROUP BY polity_id
    '''):
        pid_periods[row[0]] = (row[1], row[2])
    
    # Build metadata dataframe
    rows = []
    for _, r in lowest_level_counts.iterrows():
        pid = r['max_pid']
        if pid in pid_periods:
            start, end = pid_periods[pid]
            if start is not None and end is not None and end < MODERN_CUTOFF:
                duration = end - start
                if duration >= MIN_DURATION:
                    rows.append({
                        'polity': r['lowest_polity'],
                        'start_yr': start,
                        'collapse_yr': end,
                        'duration': duration,
                        'n_total': r['count'],
                        'polity_id': pid,
                        'type': 'long' if duration >= DURATION_THRESHOLD else 'short'
                    })
    
    df_polity_meta = pd.DataFrame(rows)
    save_checkpoint('df_polity_meta', df_polity_meta)

print(f'Non-modern polities (≥{MIN_DURATION} years): {len(df_polity_meta)}')

# ── 1c. Count individuals in windows and filter ──
df_polities = load_checkpoint('df_polities')
if df_polities is None:
    print("Counting individuals in windows...")
    polity_years = df_raw.groupby('lowest_polity')['impact_year'].apply(list).to_dict()
    
    rows = []
    for _, p in tqdm(df_polity_meta.iterrows(), total=len(df_polity_meta), desc="Filtering"):
        years = polity_years.get(p['polity'], [])
        end = p['collapse_yr']
        
        n_crisis = sum(1 for y in years if end - CRISIS_WINDOW <= y <= end)
        n_pre = sum(1 for y in years if end - PRE_CRISIS_GAP - PRE_CRISIS_WINDOW <= y < end - PRE_CRISIS_GAP)
        
        if n_crisis + n_pre >= MIN_INDIVIDUALS:
            rows.append({**p.to_dict(), 'n_crisis': n_crisis, 'n_pre_crisis': n_pre})
    
    df_polities = pd.DataFrame(rows)
    save_checkpoint('df_polities', df_polities)

print(f"Polities meeting criteria: {len(df_polities)}")

# ── 1d. Compute polity centroids ──
df_coords = load_checkpoint('df_coords')
if df_coords is None:
    print("Computing polity centroids...")
    
    pids_needed = df_polity_meta['polity_id'].unique().tolist()
    pid_to_polity = df_polity_meta.set_index('polity_id')['polity'].to_dict()
    
    geom_query = conn.execute(f'''
        SELECT polity_id, geometry FROM (
            SELECT polity_id, geometry, ROW_NUMBER() OVER (PARTITION BY polity_id ORDER BY to_year DESC) as rn
            FROM cliopatria_polity_periods 
            WHERE polity_id IN ({','.join(map(str, pids_needed))}) AND geometry IS NOT NULL
        ) WHERE rn = 1
    ''')
    
    rows = []
    for pid, geom_str in tqdm(geom_query, desc="Parsing geometries", total=len(pids_needed)):
        polity = pid_to_polity.get(pid)
        if polity:
            try:
                geom = json.loads(geom_str)
                lat, lon = get_largest_polygon_centroid(geom)
                if lat is not None:
                    rows.append({'polity': polity, 'lat': lat, 'lon': lon})
            except:
                continue
    
    df_coords = pd.DataFrame(rows)
    save_checkpoint('df_coords', df_coords)

# Merge coords into df_polities
df_polities = df_polities.merge(df_coords, on='polity', how='left')
print(f"With coordinates: {df_polities['lat'].notna().sum()}")

# ── 1e. Find 5 nearest neighbours ACTIVE AT TIME OF COLLAPSE ──
df_neighbours = load_checkpoint('df_neighbours')
if df_neighbours is None:
    print("Finding contemporary neighbours...")
    
    # All polities with coords and time periods
    all_meta = df_polity_meta.merge(df_coords, on='polity', how='inner')
    
    coord_pnames = all_meta['polity'].tolist()
    coord_array = all_meta[['lat', 'lon']].values
    meta_array = all_meta[['start_yr', 'collapse_yr']].values
    
    rows = []
    for _, p in tqdm(df_polities.iterrows(), total=len(df_polities), desc="Computing neighbours"):
        if pd.isna(p['lat']):
            continue
        
        lat, lon = p['lat'], p['lon']
        collapse_yr = p['collapse_yr']
        pname = p['polity']
        
        # Polities active at collapse time (excluding self)
        active_mask = (meta_array[:, 0] <= collapse_yr) & (meta_array[:, 1] >= collapse_yr)
        active_mask &= np.array([n != pname for n in coord_pnames])
        
        if not active_mask.any():
            continue
        
        active_indices = np.where(active_mask)[0]
        active_coords = coord_array[active_indices]
        
        dists = np.sqrt((active_coords[:, 0] - lat)**2 +
                        ((active_coords[:, 1] - lon) * np.cos(np.radians(lat)))**2)
        
        idx5 = np.argsort(dists)[:5]
        neighbours = [coord_pnames[active_indices[i]] for i in idx5]
        
        rows.append({'polity': pname, 'neighbours': ', '.join(neighbours), 'n_neighbours': len(neighbours)})
    
    df_neighbours = pd.DataFrame(rows)
    save_checkpoint('df_neighbours', df_neighbours)

# Merge neighbours
df_polities = df_polities.drop(columns=['neighbours', 'n_neighbours'], errors='ignore')
df_polities = df_polities.merge(df_neighbours, on='polity', how='left')
df_polities['n_neighbours'] = df_polities['n_neighbours'].fillna(0).astype(int)

conn.close()

# Sort by n_total
df_polities = df_polities.sort_values('n_total', ascending=False).reset_index(drop=True)

n_short = (df_polities['type'] == 'short').sum()
n_long  = (df_polities['type'] == 'long').sum()
print(f'\nPolities: {len(df_polities)} total — {n_short} short-lived, {n_long} long-lasting')
print(f'With neighbours: {(df_polities["n_neighbours"] > 0).sum()}')

df_polities

  ✓ Loaded: df_raw.pkl
Individuals: 4,204,216
  ✓ Loaded: df_polity_meta.pkl
Non-modern polities (≥26 years): 821
  ✓ Loaded: df_polities.pkl
Polities meeting criteria: 101
  ✓ Loaded: df_coords.pkl
With coordinates: 101
  ✓ Loaded: df_neighbours.pkl

Polities: 101 total — 70 short-lived, 31 long-lasting
With neighbours: 100


,polity,start_yr,collapse_yr,duration,n_total,polity_id,type,n_crisis,n_pre_crisis,lat,lon,neighbours,n_neighbours
0,Union of Soviet Socialist Republics,1922,1991,69,210476,1403,short,52467,30488,59.437888,98.979486,"Mongolian People's Republic, Czechoslovakia, S...",5
1,German Democratic Republic,1949,1990,41,92254,1464,short,29346,20455,52.388571,12.363431,"Federal Republic of Germany, Czechoslovakia, S...",5
2,German Empire,1871,1919,48,63410,1348,short,22513,11452,51.588933,12.157569,"Czechoslovakia, Free City of Danzig, Hungarian...",5
3,French Third Republic,1870,1939,69,62314,1346,short,11879,10908,46.587660,2.008573,"Kingdom of Monaco, Francoist Spain, Czechoslov...",5
4,Russian Empire,1721,1916,195,51480,1125,long,12629,6229,60.195390,99.317244,"Kingdom of Italy, Emirate of Bukhara, Empire o...",5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Albania,1912,1945,33,284,1375,short,77,81,41.072679,19.917182,"Yugoslavia, Kingdom of Romania, Hungarian Nati...",5
97,Duchy of Brunswick,1814,1870,56,283,1242,short,83,54,51.910709,10.091151,"Principality of Lippe, Principality of Waldeck...",5
98,Kingdom of Afghanistan,1926,1972,46,230,1409,short,88,38,34.235957,67.335702,"Pahlavi Dynasty, State of Kuwait, North Yemen,...",5
99,British Cape Colony,1796,1884,88,206,1173,short,96,18,-31.215042,23.451459,"Free Orange State, Natalia Republic, South Afr...",5


In [53]:
# ── 2. Compute occupation shares: crisis decade vs 2 decades before ──
# Crisis window: [collapse_yr - 10, collapse_yr]
# Pre-crisis window: [collapse_yr - 30, collapse_yr - 20]

df_shares = load_checkpoint('df_shares')
if df_shares is None:
    print("Loading occupation data...")
    conn = sqlite3.connect(DB)
    df_occ = pd.read_sql_query('''
        SELECT polity_name, polity_id, impact_year, occupations
        FROM consolidate
        WHERE impact_year IS NOT NULL AND occupations IS NOT NULL
    ''', conn)
    conn.close()

    # Extract lowest-level polity
    print("Extracting lowest-level polities...")
    df_occ['lowest_polity'] = df_occ['polity_name'].apply(extract_lowest_level_polity)
    df_occ = df_occ[df_occ['lowest_polity'].notna()].copy()

    # Keep only polities in our analysis set
    selected_polities = set(df_polities['polity'])
    df_occ = df_occ[df_occ['lowest_polity'].isin(selected_polities)].copy()
    print(f"Individuals in selected polities: {len(df_occ):,}")

    # Merge with polity info to get collapse_yr
    polity_info = df_polities[['polity', 'collapse_yr']].copy()
    df_occ = df_occ.merge(polity_info, left_on='lowest_polity', right_on='polity', how='inner')

    # Assign period using decade-based windows
    print("Assigning periods...")
    df_occ['period'] = np.select(
        [
            (df_occ['impact_year'] >= df_occ['collapse_yr'] - CRISIS_WINDOW) & 
            (df_occ['impact_year'] <= df_occ['collapse_yr']),
            (df_occ['impact_year'] >= df_occ['collapse_yr'] - PRE_CRISIS_GAP - PRE_CRISIS_WINDOW) & 
            (df_occ['impact_year'] < df_occ['collapse_yr'] - PRE_CRISIS_GAP)
        ],
        ['crisis', 'pre_crisis'],
        default=None
    )

    # Keep only rows in analysis windows
    df_occ = df_occ[df_occ['period'].notna()].copy()
    print(f"Individuals in analysis windows: {len(df_occ):,}")

    # Explode occupations
    print("Exploding occupations...")
    df_occ['occupation'] = df_occ['occupations'].str.split('; ')
    df_occ = df_occ.explode('occupation')
    df_occ['occupation'] = df_occ['occupation'].str.strip()
    df_occ = df_occ[df_occ['occupation'] != ''].copy()
    print(f"Individual-occupation pairs: {len(df_occ):,}")

    # Count occupations per polity (total across both periods) to filter >= 10
    print("Filtering occupations (≥10 per polity)...")
    occ_counts_total = df_occ.groupby(['lowest_polity', 'occupation']).size().reset_index(name='total_count')
    occ_counts_total = occ_counts_total[occ_counts_total['total_count'] >= 10]
    
    # Merge to filter
    df_occ = df_occ.merge(occ_counts_total[['lowest_polity', 'occupation']], 
                          on=['lowest_polity', 'occupation'], how='inner')
    print(f"After filtering: {len(df_occ):,} pairs")

    # Count by polity, occupation, period
    print("Computing shares...")
    counts = df_occ.groupby(['lowest_polity', 'occupation', 'period']).size().reset_index(name='count')
    totals = df_occ.groupby(['lowest_polity', 'period']).size().reset_index(name='total')
    counts = counts.merge(totals, on=['lowest_polity', 'period'])
    counts['share'] = counts['count'] / counts['total']

    # Pivot to wide format
    df_shares = counts.pivot_table(
        index=['lowest_polity', 'occupation'],
        columns='period',
        values=['count', 'share', 'total'],
        fill_value=0
    ).reset_index()

    # Flatten column names
    df_shares.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in df_shares.columns]

    # Compute changes
    df_shares['share_change'] = df_shares['share_crisis'] - df_shares['share_pre_crisis']
    df_shares['share_pct_change'] = ((df_shares['share_crisis'] - df_shares['share_pre_crisis']) / 
                                      df_shares['share_pre_crisis'].replace(0, np.nan) * 100)

    # Rename columns
    df_shares = df_shares.rename(columns={
        'lowest_polity': 'polity',
        'count_pre_crisis': 'n_pre_crisis',
        'count_crisis': 'n_crisis',
        'total_pre_crisis': 'total_pre_crisis',
        'total_crisis': 'total_crisis'
    })

    # Sort by absolute share change
    df_shares = df_shares.sort_values('share_change', key=abs, ascending=False).reset_index(drop=True)
    
    save_checkpoint('df_shares', df_shares)

print(f"\nOccupation shares:")
print(f"  Polities: {df_shares['polity'].nunique()}")
print(f"  Occupations: {df_shares['occupation'].nunique()}")
print(f"  Total rows: {len(df_shares):,}")

df_shares

  ✓ Loaded: df_shares.pkl

Occupation shares:
  Polities: 101
  Occupations: 1202
  Total rows: 7,827


,polity,occupation,n_crisis,n_pre_crisis,share_crisis,share_pre_crisis,total_crisis,total_pre_crisis,share_change,share_pct_change
0,Burma,politician,372.0,20.0,0.726562,0.232558,512.0,86.0,4.940044e-01,212.421875
1,Rattanakosin Kingdom,politician,36.0,5.0,0.837209,0.357143,43.0,14.0,4.800664e-01,134.418605
2,Rattanakosin Kingdom,civil servant,7.0,9.0,0.162791,0.642857,43.0,14.0,-4.800664e-01,-74.677003
3,Brandenburg-Prussia,pastor,0.0,14.0,0.000000,0.388889,0.0,36.0,-3.888889e-01,-100.000000
4,State of Kuwait,politician,45.0,36.0,0.267857,0.642857,168.0,56.0,-3.750000e-01,-58.333333
...,...,...,...,...,...,...,...,...,...,...
7822,German Democratic Republic,herpetologist,8.0,6.0,0.000165,0.000165,48567.0,36291.0,-6.093461e-07,-0.368563
7823,Union of Soviet Socialist Republics,sound operator,11.0,6.0,0.000123,0.000124,89296.0,48513.0,-4.923804e-07,-0.398114
7824,Union of Soviet Socialist Republics,climatologist,11.0,6.0,0.000123,0.000124,89296.0,48513.0,-4.923804e-07,-0.398114
7825,Union of Soviet Socialist Republics,chief executive officer,11.0,6.0,0.000123,0.000124,89296.0,48513.0,-4.923804e-07,-0.398114


In [54]:
# ── 3. Wilcoxon Signed-Rank Test ──
# Test if occupation share changes are systematically different from zero across polities

from scipy.stats import wilcoxon

MIN_POLITIES = 8  # minimum polities needed for Wilcoxon test

# Group share changes by occupation
occ_groups = df_shares.groupby('occupation')['share_change'].apply(list).to_dict()

results = []
for occ, deltas in occ_groups.items():
    deltas = [d for d in deltas if not np.isnan(d)]
    
    if len(deltas) < MIN_POLITIES:
        continue
    
    arr = np.array(deltas)
    
    # Skip if all zeros
    if np.all(arr == 0):
        continue
    
    try:
        stat, pval = wilcoxon(arr)
    except ValueError:
        continue
    
    mean_d = np.mean(arr)
    median_d = np.median(arr)
    
    results.append({
        'occupation': occ,
        'direction': 'RISE' if mean_d > 0 else 'FALL',
        'n_polities': len(deltas),
        'mean_delta': mean_d,
        'median_delta': median_d,
        'pvalue': pval,
        'consistency': np.mean([(d > 0) == (mean_d > 0) for d in deltas]) * 100,
    })

# Sort by p-value
results.sort(key=lambda x: x['pvalue'])

# Convert to DataFrame
df_wilcoxon = pd.DataFrame(results)

n_sig = (df_wilcoxon['pvalue'] < 0.05).sum() if len(df_wilcoxon) > 0 else 0
print(f"Wilcoxon signed-rank test results:")
print(f"  Occupations tested: {len(df_wilcoxon)}")
print(f"  Significant (p < 0.05): {n_sig}")

# Show top 30
print(f"\nTop 30 collapse-associated occupations:")
df_wilcoxon.head(30)

Wilcoxon signed-rank test results:
  Occupations tested: 273
  Significant (p < 0.05): 52

Top 30 collapse-associated occupations:


,occupation,direction,n_polities,mean_delta,median_delta,pvalue,consistency
0,film producer,RISE,19,0.002785,0.001662,0.000008,94.736842
1,singer,RISE,43,0.009122,0.003515,0.000028,79.069767
2,association football player,RISE,35,0.014829,0.009116,0.000060,80.000000
3,sociologist,RISE,17,0.001849,0.000963,0.000076,94.117647
4,designer,RISE,17,0.001128,0.001103,0.000153,94.117647
5,tennis player,RISE,13,0.000706,0.000325,0.000244,100.000000
6,musician,RISE,41,0.002847,0.001307,0.000363,70.731707
7,astronomer,FALL,18,-0.000539,-0.000351,0.000534,94.444444
8,mineralogist,FALL,14,-0.000585,-0.000586,0.000854,85.714286
9,amateur wrestler,RISE,14,0.011706,0.000976,0.001221,85.714286


In [ ]:
# ── 4. Difference-in-Differences: Collapsed vs Neighbors ──
# For each collapsed polity, compare its occupation share change against the AVERAGE of its 5 neighbors
# DiD = (collapsed share change) - (average of neighbors' share changes)

print("Computing difference-in-differences...")
conn = sqlite3.connect(DB)

# Load occupation data
df_occ_all = pd.read_sql_query('''
    SELECT polity_name, polity_id, impact_year, occupations
    FROM consolidate
    WHERE impact_year IS NOT NULL AND occupations IS NOT NULL
''', conn)
conn.close()

# Extract lowest-level polity
df_occ_all['lowest_polity'] = df_occ_all['polity_name'].apply(extract_lowest_level_polity)
df_occ_all = df_occ_all[df_occ_all['lowest_polity'].notna()].copy()

def compute_share_change_for_polity(df_polity, collapse_yr):
    """Compute share change for a single polity given a collapse year reference."""
    # Assign periods
    df_polity = df_polity.copy()
    df_polity['period'] = np.select(
        [
            (df_polity['impact_year'] >= collapse_yr - CRISIS_WINDOW) & 
            (df_polity['impact_year'] <= collapse_yr),
            (df_polity['impact_year'] >= collapse_yr - PRE_CRISIS_GAP - PRE_CRISIS_WINDOW) & 
            (df_polity['impact_year'] < collapse_yr - PRE_CRISIS_GAP)
        ],
        ['crisis', 'pre_crisis'],
        default=None
    )
    
    df_polity = df_polity[df_polity['period'].notna()].copy()
    if len(df_polity) == 0:
        return None
    
    # Explode occupations
    df_polity['occupation'] = df_polity['occupations'].str.split('; ')
    df_polity = df_polity.explode('occupation')
    df_polity['occupation'] = df_polity['occupation'].str.strip()
    df_polity = df_polity[df_polity['occupation'] != ''].copy()
    
    if len(df_polity) == 0:
        return None
    
    # Compute shares
    counts = df_polity.groupby(['occupation', 'period']).size().reset_index(name='count')
    totals = df_polity.groupby('period').size().reset_index(name='total')
    counts = counts.merge(totals, on='period')
    counts['share'] = counts['count'] / counts['total']
    
    # Pivot
    shares = counts.pivot(index='occupation', columns='period', values='share').reset_index()
    shares.columns.name = None
    
    if 'crisis' not in shares.columns or 'pre_crisis' not in shares.columns:
        return None
    
    shares['share_change'] = shares['crisis'].fillna(0) - shares['pre_crisis'].fillna(0)
    return shares[['occupation', 'share_change']].set_index('occupation')['share_change'].to_dict()

# Compute DiD for each collapsed polity
did_results = []

for _, row in tqdm(df_polities.iterrows(), total=len(df_polities), desc="Computing DiD"):
    polity = row['polity']
    collapse_yr = row['collapse_yr']
    neighbours_str = row['neighbours']
    
    if pd.isna(neighbours_str) or not neighbours_str:
        continue
    
    neighbours = [n.strip() for n in neighbours_str.split(',')]
    
    # Get collapsed polity's share changes
    collapsed_shares = df_shares[df_shares['polity'] == polity].set_index('occupation')['share_change'].to_dict()
    
    if not collapsed_shares:
        continue
    
    # Compute share change for EACH neighbor separately
    neighbor_changes = {}  # occupation -> list of changes from each neighbor
    
    for neighbor in neighbours:
        df_neighbor = df_occ_all[df_occ_all['lowest_polity'] == neighbor].copy()
        if len(df_neighbor) == 0:
            continue
        
        neighbor_share_changes = compute_share_change_for_polity(df_neighbor, collapse_yr)
        if neighbor_share_changes is None:
            continue
        
        for occ, change in neighbor_share_changes.items():
            if occ not in neighbor_changes:
                neighbor_changes[occ] = []
            neighbor_changes[occ].append(change)
    
    # Compute DiD: collapsed - average(neighbors)
    for occ, collapsed_change in collapsed_shares.items():
        if occ not in neighbor_changes or len(neighbor_changes[occ]) == 0:
            continue
        
        avg_neighbor_change = np.mean(neighbor_changes[occ])
        did = collapsed_change - avg_neighbor_change
        
        did_results.append({
            'polity': polity,
            'occupation': occ,
            'collapsed_share_change': collapsed_change,
            'avg_neighbor_change': avg_neighbor_change,
            'n_neighbors': len(neighbor_changes[occ]),
            'did': did
        })

# Combine all results
df_did = pd.DataFrame(did_results)
print(f"\nDiD results: {len(df_did):,} polity-occupation pairs")
print(f"Average neighbors per comparison: {df_did['n_neighbors'].mean():.1f}")

# Save checkpoint
save_checkpoint('df_did', df_did)

df_did

In [56]:
# ── 5. Wilcoxon Test on DiD Estimates ──
# Test if DiD (collapse-specific signal) is systematically different from zero

MIN_POLITIES = 8

# Group DiD by occupation
did_groups = df_did.groupby('occupation')['did'].apply(list).to_dict()

results_did = []
for occ, deltas in did_groups.items():
    deltas = [d for d in deltas if not np.isnan(d)]
    
    if len(deltas) < MIN_POLITIES:
        continue
    
    arr = np.array(deltas)
    
    if np.all(arr == 0):
        continue
    
    try:
        stat, pval = wilcoxon(arr)
    except ValueError:
        continue
    
    mean_d = np.mean(arr)
    median_d = np.median(arr)
    
    results_did.append({
        'occupation': occ,
        'direction': 'RISE' if mean_d > 0 else 'FALL',
        'n_polities': len(deltas),
        'mean_did': mean_d,
        'median_did': median_d,
        'pvalue': pval,
        'consistency': np.mean([(d > 0) == (mean_d > 0) for d in deltas]) * 100,
    })

# Sort by p-value
results_did.sort(key=lambda x: x['pvalue'])

# Convert to DataFrame
df_wilcoxon_did = pd.DataFrame(results_did)

# Compare with raw results
if len(df_wilcoxon) > 0:
    raw_rank = {r: i+1 for i, r in enumerate(df_wilcoxon['occupation'])}
    df_wilcoxon_did['raw_rank'] = df_wilcoxon_did['occupation'].map(raw_rank)

n_sig = (df_wilcoxon_did['pvalue'] < 0.05).sum() if len(df_wilcoxon_did) > 0 else 0
print(f"DiD Wilcoxon signed-rank test results:")
print(f"  Occupations tested: {len(df_wilcoxon_did)}")
print(f"  Significant (p < 0.05): {n_sig}")

# Show top 30
print(f"\nTop 30 collapse-specific signals (DiD-controlled):")
df_wilcoxon_did.head(30)

DiD Wilcoxon signed-rank test results:
  Occupations tested: 260
  Significant (p < 0.05): 19

Top 30 collapse-specific signals (DiD-controlled):


,occupation,direction,n_polities,mean_did,median_did,pvalue,consistency,raw_rank
0,graphic artist,RISE,27,0.003433,0.003047,0.003878,77.777778,53
1,mineralogist,FALL,12,-0.001016,-0.000328,0.012207,83.333333,9
2,children's writer,FALL,13,-0.000825,-0.000343,0.013428,84.615385,161
3,designer,RISE,17,0.000921,0.000433,0.017426,76.470588,5
4,dramaturge,RISE,11,0.002023,0.001164,0.024414,90.909091,190
5,Catholic bishop,FALL,39,-0.005458,-0.001611,0.026804,66.666667,51
6,philosopher,RISE,40,0.003606,0.000698,0.026853,57.500000,40
7,music teacher,FALL,10,-0.000589,-0.000286,0.027344,90.000000,168
8,school teacher,FALL,14,-0.001345,-0.000374,0.029541,78.571429,126
9,illustrator,RISE,27,0.001520,0.001873,0.033995,66.666667,123
